# SpendShield — Revised Synthetic Dataset Generation

This notebook creates dataset version `v2` with realistic behavioral overlap.
It is research-only: labels are generator-defined synthetic scenarios, not
fraud facts, and no MongoDB/Cassandra/backend state is accessed or changed.


## Objective and reproducibility

Version `v1` is preserved. Version `v2` uses generator `1.1.0`, seed `20260913`,
and varied normal, amount, temporal, merchant, category, and channel behavior.
The output remains bounded, fictional, INR-only, and timestamp-split.


In [1]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
for candidate in (ROOT, ROOT.parent, ROOT.parent.parent):
    if (candidate / "ml").is_dir() and (candidate / "data" / "synthetic").is_dir():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

V2_DIR = ROOT / "data" / "synthetic" / "v2"
V2_FEATURE_DIR = V2_DIR / "features"
V2_BASELINE_DIR = V2_DIR / "baseline"
V2_ERROR_DIR = V2_DIR / "error_analysis"
V2_COMPARISON_DIR = ROOT / "data" / "synthetic" / "comparison"
from ml.synthetic_dataset_generator import (
    generate_revised_dataset,
    revised_dataset_config,
    validate_written_dataset,
    write_dataset,
)

config = revised_dataset_config()
dataset = generate_revised_dataset(config)
write_dataset(dataset, V2_DIR, config)
validation = validate_written_dataset(V2_DIR)
print(json.dumps({
    "valid": validation["valid"],
    "dataset_version": dataset.manifest["dataset_version"],
    "generator_version": dataset.manifest["generator_version"],
    "seed": dataset.manifest["random_seed"],
    "row_count": validation["row_count"],
    "split_counts": validation["split_counts"],
    "scenario_distribution": validation["scenario_distribution"],
}, indent=2))


{
  "valid": true,
  "dataset_version": "v2",
  "generator_version": "1.1.0",
  "seed": 20260913,
  "row_count": 10000,
  "split_counts": {
    "train": 7328,
    "validation": 1474,
    "test": 1198
  },
  "scenario_distribution": {
    "normal": 6800,
    "synthetic_behavior_deviation": 500,
    "synthetic_combined_pattern": 200,
    "synthetic_high_amount": 1000,
    "synthetic_rapid_repeat": 800,
    "synthetic_unusual_time": 700
  }
}


In [2]:
manifest = json.loads((V2_DIR / "dataset_manifest.json").read_text(encoding="utf-8"))
print("Date range:", manifest["date_range"])
print("Currency: INR")
print("Provenance:", manifest["dataset_type"], manifest["label_notice"])


Date range: {'configured_end_exclusive': '2024-03-31T00:00:00Z', 'configured_start': '2024-01-01T00:00:00Z', 'end': '2024-03-27T23:46:46Z', 'start': '2024-01-01T00:07:54Z', 'timestamp_precision': 'second', 'timezone': 'UTC'}
Currency: INR
Provenance: synthetic_research Synthetic scenario labels identify intentionally generated scenarios. They do not represent confirmed fraud, real financial crime, or real-world risk outcomes.


The generated artifact is deliberately not described as fraud data.
The next notebook validates overlap, schema, temporal integrity, and feature
boundaries before any baseline is re-evaluated.
